In [ ]:
import pandas as pd
from utils.config import load_config
from utils.shared import get_model_by_name, save_model
from utils.training import get_tuned_hyperparameters, train_model


In [ ]:
config = load_config()


In [ ]:
for model_name in config.models:
    for data_dim in config.sample_sizes:

        print(
            f"Start training {model_name} model for {data_dim}x{config.train_samples_num} data..."
        )

        if model_name not in ["DFA", "RS"]:
            hyperparams = get_tuned_hyperparameters(model_name, data_dim)
        else:
            hyperparams = {}
        estimator = get_model_by_name(
            model_name, config.random_seed, config.n_jobs, input_shape=(data_dim, 1)
        )
        data = pd.read_csv(
            f"{config.data_dir}/{config.train_data_subdir}/fbm_{data_dim}x{config.train_samples_num}.csv"
        )

        model = train_model(
            experiment_name=config.training_experiment_name,
            run_name=f"{model_name}_{data_dim}",
            estimator=estimator,
            hyperparams=hyperparams,
            X_train=data.drop(columns=[config.target_column]),
            y_train=data[config.target_column],
        )

        save_model(model, f"models/{model_name}_{data_dim}.joblib")
